<h1>PPDB Export Examples</h1>

<h2>Imports</h2>

In [ ]:
from pathlib import Path
import io
import requests
import time

from astropy.table import Table
from pyvo.dal import AsyncTAPJob, TAPService
from pyvo.dal.tap import TAPService
import pandas as pd
import pyvo

from lsst.rsp import RSPClient, get_tap_service, get_service_url, get_access_token

<h2>Service setup</h2>

Get the PPDB TAP service.

In [ ]:
url = get_service_url("tap", "prompt")

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {get_access_token()}"
})
service = TAPService(url, session=session)

<h2>Export utility function</h2>

This method will query a specific PPDB table using the TAP service to determine which days have data and what are the expected record counts. The data will then be exported to a set of parquet files, one per day. If a parquet file already exists with the expected number of records in the output directory, the export for that day will be skipped unless `skip_existing` is set to `False`.

In [ ]:
def export_ppdb_table(
    table_name: str,
    export_dir: str = "ppdb_export_data",
    skip_existing: bool = True
):
    """Export PPDB table data to parquet files, one per day.

    Days will be skipped if there is a parquet file already present 
    with the correct record count.

    Parameters
    ----------
    table_name
        Name of table to export such as "DiaObject" or "DiaSource".
    export_dir
        Directory where parquet files should be written.
        Defaults to ``ppdb_export_data.
    """
    print(f"Starting export of {table_name} table...\n")
    export_start = time.time()
    
    # Create export directory (or use existing).
    Path(export_dir).mkdir(exist_ok=True)

    # Determine which timing column to use for the table.
    if table_name == "DiaObject":
        ts_col = "validityStartMjdTai"
    elif table_name == "DiaSource" or table_name == "DiaForcedSource":
        ts_col = "midpointMjdTai"
    else:
        raise Exception(f"Unsupported table: {table_name}")

    # Get a list of days (MJD TAI format) which have data.
    sql = f"""
        SELECT FLOOR({ts_col}) AS day_mjd_tai,
            COUNT(*) AS record_count
        FROM ppdb.{table_name}
        GROUP BY day_mjd_tai ORDER BY day_mjd_tai
        """
    job = service.submit_job(sql)
    job.run()
    job.wait(phases=['COMPLETED', 'ERROR'])
    if job.phase == "ERROR":
        job.raise_if_error() 
    days_result = job.fetch_result().to_table()
    
    days = [d for d in days_result["day_mjd_tai"]]
    record_counts = [c for c in days_result["record_count"]]
    
    # Loop over the days with data and process them.
    for day, record_count in zip(days, record_counts, strict=True):

        print(f"Processing day: {day}")
        print(f"  Expected record count: {record_count}")

        day_start = time.time()
        
        # Make directory for this day.
        output_dir = Path(export_dir, str(int(day)))
        output_dir.mkdir(exist_ok=True)
        output_path = output_dir / f"{table_name}.parquet"
    
        # Check for and verify an existing output file and skip if exists
        # with correct record count.
        if skip_existing and output_path.exists():
            print(f"  Parquet file already exists: {output_path}")
            try:
                df = pd.read_parquet(output_path)
                parquet_record_count = len(df)
                print(f"  Existing parquet file has {parquet_record_count} records.")
                if parquet_record_count == record_count:
                    print("  Skipping this day - parquet file with correct record count already exists.\n")
                    continue
                else:
                    print(f"  Record count mismatch: {parquet_record_count} != {record_count}")
                    print("  File will be recreated.")
            except Exception as e:
                # This probably indicates an invalid or partially written parquet file.
                print(e)
        
        # Get data for the day from the TAP service.
        sql = f"SELECT * FROM ppdb.{table_name} WHERE FLOOR({ts_col}) = {day}"
        print(f"  Executing SQL: {sql}")
         
        # Run the SQL job.
        job_start = time.time()
        job = AsyncTAPJob.create(
            service.baseurl,
            sql,
            RESPONSEFORMAT="application/vnd.apache.parquet",
            session=session
        )
        job = job.run().wait()
        job_end = time.time()
        job_elapsed = job_end - job_start
        print(f"  Job took {job_elapsed:.2f} seconds")

        # Fetch the data.
        fetch_start = time.time()
        response = session.get(job.result_uri, stream=True)
        table_data = Table.read(io.BytesIO(response.content), format="parquet.votable")
        fetch_end = time.time() 
        fetch_elapsed = fetch_end - fetch_start
        print(f"  Fetch took {fetch_elapsed:.2f} seconds")
            
        # Write the entire day's data to a parquet file.
        parq_start = time.time()
        table_data.write(output_path, format="parquet", overwrite=True)
        parq_end = time.time()
        parq_elapsed = parq_end - parq_start
        print(f"  Wrote table data to '{output_path}' in {parq_elapsed:.2f} seconds")

        day_end = time.time()
        day_elapsed = day_end - day_start
        print(f"  Exported data from day {day} in {day_elapsed:.2f} seconds\n")

    export_end = time.time()
    export_elapsed = export_end - export_start
    print(f"Export of {table_name} completed in {export_elapsed:.0f} seconds.")

<h2>Export table data</h2>

In [ ]:
for table_name in ["DiaObject", "DiaSource", "DiaForcedSource"]:
    export_ppdb_table(table_name, skip_existing=True)